# 3.1.0 Significance Testing

This notebook loads the saved drift metrics and aligned corpus metadata, runs grouped permutation significance tests for pooled and category-specific drift, and exports significance-aware tables and figures.


## Purpose

Load the saved drift metrics and aligned corpus metadata, run grouped permutation significance tests for raw and category-specific drift, and export significance-aware tables and figures.

---

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
NOTEBOOKS_DIR = CODE_DIR / "notebooks"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

# Repo mode omits manuscript and Overleaf export surfaces.
def record_figure_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

def record_table_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

import gc
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dcor import energy_distance
from matplotlib.lines import Line2D
from scipy.stats import wasserstein_distance, spearmanr, kendalltau
from itertools import permutations

from notebook_theme import CAT_LABELS, activate_theme, colors, style_3d_axis, style_axis, themes

FORCE_RERUN_SIG = True
SEED = 42
RUN_ID = "analysis_corpus"
RUN = "final"

RUN_CONFIG = {
    "test": {
        "n_perm": 99,
        "target_rows_default": 600,
        "min_rows_for_test": 200,
        "mmd_max_rows": 400,
        "ed_max_rows": 400,
        "sw_max_rows": 600,
        "sw_n_proj": 32,
        "subspace_max_rows": 600,
        "subspace_n_components": 20,
        "sigma_sample_rows": 1000,
        "sigma_pair_rows": 200,
    },
    "final": {
        "n_perm": 999,
        "target_rows_default": 1200,
        "min_rows_for_test": 400,
        "mmd_max_rows": 800,
        "ed_max_rows": 800,
        "sw_max_rows": 1200,
        "sw_n_proj": 64,
        "subspace_max_rows": 1200,
        "subspace_n_components": 20,
        "sigma_sample_rows": 2000,
        "sigma_pair_rows": 450,
    },
}
run_cfg = RUN_CONFIG[RUN]

N_PERM = run_cfg["n_perm"]
TARGET_ROWS_DEFAULT = run_cfg["target_rows_default"]
MIN_ROWS_FOR_TEST = run_cfg["min_rows_for_test"]
MMD_MAX_ROWS = run_cfg["mmd_max_rows"]
ED_MAX_ROWS = run_cfg["ed_max_rows"]
SW_MAX_ROWS = run_cfg["sw_max_rows"]
SW_N_PROJ = run_cfg["sw_n_proj"]
SUBSPACE_MAX_ROWS = run_cfg["subspace_max_rows"]
SUBSPACE_N_COMPONENTS = run_cfg["subspace_n_components"]
SIGMA_SAMPLE_ROWS = run_cfg["sigma_sample_rows"]
SIGMA_PAIR_ROWS = run_cfg["sigma_pair_rows"]

REDUCER = "ae"
REDUCED_DIM = 200
REDUCED_LAYER_MODE = "all_mean"
REDUCED_LABEL = f"{REDUCER.upper()}-{REDUCED_DIM}_{REDUCED_LAYER_MODE}"
SIG_LABEL_MODE = "dominant"
SIG_USE_DOMINANT = True
SIG_CATEGORIES = ("A", "B")
THEME_NAME = "paper"

theme = activate_theme(THEME_NAME)

RUN_ROOT = OUTPUTS_DIR / "analytical-results" / RUN_ID
CORPUS_ROOT = LOCAL_DIR / "corpora" / RUN_ID
CORPUS_BINS_DIR = CORPUS_ROOT / "bins"
CORPUS_STAGE_DIR = RUN_ROOT / "corpus"
CORPUS_DATA_DIR = CORPUS_STAGE_DIR / "data"
DRIFT_DATA_DIR = RUN_ROOT / "drift" / "data"
SIGNIFICANCE_DATA_DIR = RUN_ROOT / "significance" / "data"
SIGNIFICANCE_PLOT_DIR = RUN_ROOT / "significance" / "plots" / THEME_NAME

for path in [SIGNIFICANCE_DATA_DIR, SIGNIFICANCE_PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

INDEX_PATH = CORPUS_DATA_DIR / "embedding_label_index.jsonl"
CLASSIFIED_PATH = CORPUS_STAGE_DIR / "corpus_classified.jsonl"
RAW_DRIFT_PATH = DRIFT_DATA_DIR / f"drift_metrics_raw_{RUN}.csv"
CAT_DRIFT_PATH = DRIFT_DATA_DIR / f"drift_metrics_{SIG_LABEL_MODE}_{RUN}.csv"
OUT_CSV_RAW = SIGNIFICANCE_DATA_DIR / f"drift_metrics_raw_{RUN}_with_sig.csv"
SIG_CONFIG_PATH_RAW = SIGNIFICANCE_DATA_DIR / f"drift_metrics_raw_{RUN}_with_sig_config.json"
OUT_CSV_CAT = SIGNIFICANCE_DATA_DIR / f"drift_metrics_{SIG_LABEL_MODE}_{RUN}_with_sig.csv"
SIG_CONFIG_PATH_CAT = SIGNIFICANCE_DATA_DIR / f"drift_metrics_{SIG_LABEL_MODE}_{RUN}_with_sig_config.json"
REDUCTION_ROOT = OUTPUTS_DIR / "dimension-reduction" / "models" / RUN_ID
REDUCED_FILE = REDUCTION_ROOT / REDUCER / f"{REDUCER}_{REDUCED_DIM}d_{REDUCED_LAYER_MODE}.npy"

print(f"RUN mode: {RUN}")
print(run_cfg)
print("RUN_ROOT:", RUN_ROOT)
print("CORPUS_BINS_DIR:", CORPUS_BINS_DIR, "exists:", CORPUS_BINS_DIR.exists())
print("CLASSIFIED_PATH:", CLASSIFIED_PATH, "exists:", CLASSIFIED_PATH.exists())
print("INDEX_PATH:", INDEX_PATH, "exists:", INDEX_PATH.exists())
print("RAW_DRIFT_PATH:", RAW_DRIFT_PATH, "exists:", RAW_DRIFT_PATH.exists())
print("CAT_DRIFT_PATH:", CAT_DRIFT_PATH, "exists:", CAT_DRIFT_PATH.exists())
print("REDUCED_FILE:", REDUCED_FILE, "exists:", REDUCED_FILE.exists())

for required_path in [CLASSIFIED_PATH, INDEX_PATH, RAW_DRIFT_PATH, CAT_DRIFT_PATH, REDUCED_FILE]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")


## Definitions, functions, helpers
---

In [ ]:
# ──────────────────────────────────────────────────────────────────────
def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)

    mask = np.isfinite(pvals)
    if mask.sum() == 0:
        return out

    p = pvals[mask]
    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)

    q = ranked * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    out_mask = np.empty_like(q)
    out_mask[order] = q
    out[mask] = out_mask
    return out

# ──────────────────────────────────────────────────────────────────────
def build_row_meta(corpus_df):
    """
    corpus_df is already the row-aligned metadata table loaded from
    embedding_label_index.jsonl, so no additional reconciliation is needed.
    """
    required = ["sent", "year", "bin", "bibcode", "labels", "dominant"]
    missing = [c for c in required if c not in corpus_df.columns]
    if missing:
        raise ValueError(f"corpus_df is missing required columns: {missing}")

    row_meta = corpus_df.copy().reset_index(drop=True)
    row_meta["row_id"] = np.arange(len(row_meta), dtype=int)

    return row_meta[["row_id", "sent", "year", "bin", "bibcode", "labels", "dominant"]], "aligned-index"

# ──────────────────────────────────────────────────────────────────────
def build_group_rows(row_meta, bin_labels):
    out = {}
    for b in bin_labels:
        subset = row_meta.loc[row_meta["bin"] == b, ["row_id", "bibcode"]]
        out[b] = {
            bibcode: grp["row_id"].to_numpy(dtype=int)
            for bibcode, grp in subset.groupby("bibcode", sort=False)
        }
    return out

# ──────────────────────────────────────────────────────────────────────
def build_category_group_rows(row_meta, bin_labels, categories, use_dominant=True):
    group_rows = {cat: {} for cat in categories}
    row_counts = {cat: {} for cat in categories}
    bib_counts = {cat: {} for cat in categories}

    for cat in categories:
        for b in bin_labels:
            subset = row_meta[row_meta["bin"] == b]
            if use_dominant:
                subset = subset[subset["dominant"] == cat]
            else:
                subset = subset[subset["labels"].apply(lambda xs: cat in xs)]

            row_counts[cat][b] = int(len(subset))
            bib_counts[cat][b] = int(subset["bibcode"].nunique())
            group_rows[cat][b] = {
                bibcode: grp["row_id"].to_numpy(dtype=int)
                for bibcode, grp in subset[["row_id", "bibcode"]].groupby("bibcode", sort=False)
            }

    return group_rows, row_counts, bib_counts

# ──────────────────────────────────────────────────────────────────────
def normalize_bin_label(label, valid_labels):
    label = str(label)
    if label in valid_labels:
        return label

    try:
        start, end = map(int, label.split("-"))
    except Exception:
        return label

    candidates = []
    for valid in valid_labels:
        try:
            vstart, vend = map(int, str(valid).split("-"))
        except Exception:
            continue
        if vend == end:
            candidates.append((valid, vstart))

    if len(candidates) == 1 and abs(candidates[0][1] - start) <= 1:
        return candidates[0][0]

    return label

# ──────────────────────────────────────────────────────────────────────
def normalize_drift_bins(df_drift, valid_labels):
    df_norm = df_drift.copy()
    for col in ["bin_from", "bin_to"]:
        if col in df_norm.columns:
            df_norm[col] = df_norm[col].apply(lambda x: normalize_bin_label(x, valid_labels))
    return df_norm

# ──────────────────────────────────────────────────────────────────────
def sample_grouped_rows(group_rows, target_rows, rng):
    bibcodes = np.array(list(group_rows.keys()), dtype=object)
    order = rng.permutation(len(bibcodes))

    chosen = []
    total_rows = 0
    for j in order:
        bc = bibcodes[j]
        chosen.append(bc)
        total_rows += len(group_rows[bc])
        if total_rows >= target_rows:
            break

    if total_rows < target_rows:
        raise ValueError(f"Could only collect {total_rows} rows below target_rows={target_rows}")

    chosen_groups = {bc: group_rows[bc] for bc in chosen}
    rows = np.concatenate([chosen_groups[bc] for bc in chosen])

    if len(rows) > target_rows:
        rows = rng.choice(rows, size=target_rows, replace=False)

    return np.asarray(rows, dtype=int), chosen_groups

# ──────────────────────────────────────────────────────────────────────
def permute_group_labels(sampled_groups_1, sampled_groups_2, target_rows, rng, max_tries=1000):
    combined_groups = list(sampled_groups_1.values()) + list(sampled_groups_2.values())

    for _ in range(max_tries):
        order = rng.permutation(len(combined_groups))

        left_chunks = []
        left_rows = 0
        split = 0
        while split < len(order) and left_rows < target_rows:
            arr = combined_groups[order[split]]
            left_chunks.append(arr)
            left_rows += len(arr)
            split += 1

        right_chunks = [combined_groups[j] for j in order[split:]]
        right_rows = sum(len(arr) for arr in right_chunks)

        if left_rows >= target_rows and right_rows >= target_rows:
            left = np.concatenate(left_chunks)
            right = np.concatenate(right_chunks)

            if len(left) > target_rows:
                left = rng.choice(left, size=target_rows, replace=False)
            if len(right) > target_rows:
                right = rng.choice(right, size=target_rows, replace=False)

            return np.asarray(left, dtype=int), np.asarray(right, dtype=int)

    raise RuntimeError("Could not build a valid grouped permutation split.")

# ──────────────────────────────────────────────────────────────────────
def centroid_cosine_distance_np(X1, X2):
    c1 = X1.mean(axis=0)
    c2 = X2.mean(axis=0)
    denom = np.linalg.norm(c1) * np.linalg.norm(c2)
    if denom == 0:
        return np.nan
    return float(1.0 - np.dot(c1, c2) / denom)

# ──────────────────────────────────────────────────────────────────────
def maybe_subsample_rows(X, n_max, rng):
    if len(X) <= n_max:
        return X
    idx = rng.choice(len(X), size=n_max, replace=False)
    return X[idx]

# ──────────────────────────────────────────────────────────────────────
def mmd_rbf_fixed_sigma(X1, X2, sigma, n_max=MMD_MAX_ROWS, seed=SEED):
    rng = np.random.default_rng(seed)
    X1s = maybe_subsample_rows(X1, n_max=n_max, rng=rng)
    X2s = maybe_subsample_rows(X2, n_max=n_max, rng=rng)

    def rbf(A, B):
        dists_sq = np.sum((A[:, None] - B[None, :]) ** 2, axis=-1)
        return np.exp(-dists_sq / (2 * sigma ** 2))

    return float(rbf(X1s, X1s).mean() + rbf(X2s, X2s).mean() - 2 * rbf(X1s, X2s).mean())

# ──────────────────────────────────────────────────────────────────────
def energy_distance_capped(X1, X2, n_max=ED_MAX_ROWS, seed=SEED):
    rng = np.random.default_rng(seed)
    X1s = maybe_subsample_rows(X1, n_max=n_max, rng=rng)
    X2s = maybe_subsample_rows(X2, n_max=n_max, rng=rng)
    return float(energy_distance(X1s, X2s))

# ──────────────────────────────────────────────────────────────────────
def wasserstein_1d_projected_capped(X1, X2, n_proj=SW_N_PROJ, n_max=SW_MAX_ROWS, seed=SEED):
    rng = np.random.default_rng(seed)
    X1s = maybe_subsample_rows(X1, n_max=n_max, rng=rng)
    X2s = maybe_subsample_rows(X2, n_max=n_max, rng=rng)

    dim = X1s.shape[1]
    directions = rng.standard_normal((n_proj, dim))
    directions /= np.linalg.norm(directions, axis=1, keepdims=True)

    P1 = X1s @ directions.T
    P2 = X2s @ directions.T
    dists = [wasserstein_distance(P1[:, j], P2[:, j]) for j in range(n_proj)]
    return float(np.mean(dists))

# ──────────────────────────────────────────────────────────────────────
def subspace_drift_capped(X1, X2, n_components=SUBSPACE_N_COMPONENTS, n_max=SUBSPACE_MAX_ROWS, seed=SEED):
    rng = np.random.default_rng(seed)
    X1s = maybe_subsample_rows(X1, n_max=n_max, rng=rng)
    X2s = maybe_subsample_rows(X2, n_max=n_max, rng=rng)

    _, _, Vt1 = np.linalg.svd(X1s - X1s.mean(axis=0), full_matrices=False)
    _, _, Vt2 = np.linalg.svd(X2s - X2s.mean(axis=0), full_matrices=False)

    k = min(n_components, Vt1.shape[0], Vt2.shape[0])
    M = Vt1[:k] @ Vt2[:k].T
    sigma = np.clip(np.linalg.svd(M, compute_uv=False), -1.0, 1.0)
    return float(np.arccos(sigma).mean())

# ──────────────────────────────────────────────────────────────────────
def grouped_permutation_test(X, sampled_groups_1, sampled_groups_2, obs_rows_1, obs_rows_2, metric_fn, n_perm, seed):
    obs = metric_fn(X[obs_rows_1], X[obs_rows_2], seed=seed)

    rng = np.random.default_rng(seed)
    null = np.empty(n_perm, dtype=float)

    for i in range(n_perm):
        perm_rows_1, perm_rows_2 = permute_group_labels(
            sampled_groups_1,
            sampled_groups_2,
            target_rows=len(obs_rows_1),
            rng=rng,
        )
        null[i] = metric_fn(X[perm_rows_1], X[perm_rows_2], seed=seed + 10_000 + i)

    p = (1.0 + np.sum(null >= obs)) / (n_perm + 1.0)
    return float(obs), float(p), null

# ──────────────────────────────────────────────────────────────────────
def run_significance_branch(
    *,
    X,
    branch_name,
    categories,
    group_rows_by_cat,
    row_counts_by_cat,
    bib_counts_by_cat,
    bin_labels,
    base_drift_path,
    out_csv,
    sig_config_path,
    common_config,
):
    expected_config = {
        **common_config,
        "branch_name": branch_name,
        "categories": list(categories),
        "base_drift_path": str(base_drift_path),
        "out_csv": str(out_csv),
    }

    sig_config = None
    if not FORCE_RERUN_SIG and out_csv.exists():
        df_drift_sig = pd.read_csv(out_csv)
        print(f"Loaded cached {branch_name} significance results from {out_csv}")

        if sig_config_path.exists():
            with open(sig_config_path, "r", encoding="utf-8") as f:
                sig_config = json.load(f)
            print(f"Loaded {branch_name} significance config from {sig_config_path}")
        else:
            print(f"No cached config JSON found for {branch_name}.")

        mismatches = {}
        if sig_config is not None:
            for k, expected in expected_config.items():
                actual = sig_config.get(k)
                if actual != expected:
                    mismatches[k] = {"cached": actual, "current": expected}

        if mismatches:
            print(f"Warning: cached {branch_name} significance results were created with different settings:")
            for k, v in mismatches.items():
                print(f"  {k}: cached={v['cached']} | current={v['current']}")
            print("Set FORCE_RERUN_SIG = True if you want to recompute.")
        else:
            print(f"Cached {branch_name} significance settings match current configuration.")

        return df_drift_sig, sig_config

    metric_fns = {
        "cosine_dist": lambda A, B, seed=SEED: centroid_cosine_distance_np(A, B),
        "mmd": lambda A, B, seed=SEED: mmd_rbf_fixed_sigma(A, B, sigma=sigma_global, seed=seed),
        "sliced_wasserstein": lambda A, B, seed=SEED: wasserstein_1d_projected_capped(A, B, seed=seed),
        "energy_distance": lambda A, B, seed=SEED: energy_distance_capped(A, B, seed=seed),
        "subspace_drift": lambda A, B, seed=SEED: subspace_drift_capped(A, B, seed=seed),
    }

    results_sig = []

    for cat in categories:
        bins_available = [
            b for b in bin_labels
            if row_counts_by_cat[cat].get(b, 0) > 0 and len(group_rows_by_cat[cat].get(b, {})) > 0
        ]

        if len(bins_available) < 2:
            print(f"Skipping {branch_name}:{cat} - not enough bins with data")
            continue

        anchor_bin = bins_available[0]
        anchor_target = min(TARGET_ROWS_DEFAULT, row_counts_by_cat[cat][anchor_bin])
        if anchor_target < MIN_ROWS_FOR_TEST:
            print(
                f"Skipping {branch_name}:{cat} - anchor bin {anchor_bin} only supports "
                f"{anchor_target} rows below MIN_ROWS_FOR_TEST={MIN_ROWS_FOR_TEST}"
            )
            continue

        rng_anchor = np.random.default_rng(SEED + 50_000 + sum(ord(ch) for ch in str(cat)))
        anchor_rows_fixed, anchor_groups_fixed = sample_grouped_rows(
            group_rows_by_cat[cat][anchor_bin],
            anchor_target,
            rng_anchor,
        )

        for i in range(len(bins_available) - 1):
            b1 = bins_available[i]
            b2 = bins_available[i + 1]

            target_adj = min(TARGET_ROWS_DEFAULT, row_counts_by_cat[cat][b1], row_counts_by_cat[cat][b2])
            if target_adj < MIN_ROWS_FOR_TEST:
                print(
                    f"Skipping {branch_name}:{cat} {b1} -> {b2}: "
                    f"target_adj={target_adj} < MIN_ROWS_FOR_TEST"
                )
                continue

            seed_offset = sum(ord(ch) for ch in str(cat))
            rng_a1 = np.random.default_rng(SEED + 1_000 * i + 1 + seed_offset)
            rng_a2 = np.random.default_rng(SEED + 1_000 * i + 2 + seed_offset)

            obs_rows_1, sampled_groups_1 = sample_grouped_rows(group_rows_by_cat[cat][b1], target_adj, rng_a1)
            obs_rows_2, sampled_groups_2 = sample_grouped_rows(group_rows_by_cat[cat][b2], target_adj, rng_a2)

            rec = {
                "run_id": RUN_ID,
                "run_mode": RUN,
                "reduction": REDUCED_LABEL,
                "category": cat,
                "bin_from": b1,
                "bin_to": b2,
                "target_rows_adj": int(target_adj),
                "sampled_bibcodes_adj_from": int(len(sampled_groups_1)),
                "sampled_bibcodes_adj_to": int(len(sampled_groups_2)),
                "unique_bibcodes_from": int(bib_counts_by_cat[cat][b1]),
                "unique_bibcodes_to": int(bib_counts_by_cat[cat][b2]),
            }

            print(f"Running {branch_name} significance tests for {cat}: {b1} -> {b2} ...")

            for j, (metric_name, metric_fn) in enumerate(metric_fns.items(), start=1):
                obs, p, _ = grouped_permutation_test(
                    X=X,
                    sampled_groups_1=sampled_groups_1,
                    sampled_groups_2=sampled_groups_2,
                    obs_rows_1=obs_rows_1,
                    obs_rows_2=obs_rows_2,
                    metric_fn=metric_fn,
                    n_perm=N_PERM,
                    seed=SEED + 10_000 * i + 100 * j + seed_offset,
                )
                rec[f"{metric_name}_obs_balanced"] = obs
                rec[f"{metric_name}_p"] = p

            target_cum = min(anchor_target, row_counts_by_cat[cat][b2])
            if target_cum >= MIN_ROWS_FOR_TEST:
                rng_current = np.random.default_rng(SEED + 60_000 + i + seed_offset)
                current_rows, current_groups = sample_grouped_rows(
                    group_rows_by_cat[cat][b2],
                    target_cum,
                    rng_current,
                )

                if target_cum < len(anchor_rows_fixed):
                    rng_trim = np.random.default_rng(SEED + 65_000 + i + seed_offset)
                    anchor_rows = rng_trim.choice(anchor_rows_fixed, size=target_cum, replace=False)
                else:
                    anchor_rows = anchor_rows_fixed

                cum_obs, cum_p, _ = grouped_permutation_test(
                    X=X,
                    sampled_groups_1=anchor_groups_fixed,
                    sampled_groups_2=current_groups,
                    obs_rows_1=anchor_rows,
                    obs_rows_2=current_rows,
                    metric_fn=lambda A, B, seed=SEED: centroid_cosine_distance_np(A, B),
                    n_perm=N_PERM,
                    seed=SEED + 70_000 + i + seed_offset,
                )
                rec["cumulative_cos_obs_balanced"] = cum_obs
                rec["cumulative_cos_p"] = cum_p
            else:
                rec["cumulative_cos_obs_balanced"] = np.nan
                rec["cumulative_cos_p"] = np.nan

            print(
                f"{branch_name}:{cat} {b1} -> {b2} | "
                f"cos p={rec['cosine_dist_p']:.4f} | "
                f"mmd p={rec['mmd_p']:.4f} | "
                f"sw p={rec['sliced_wasserstein_p']:.4f} | "
                f"ed p={rec['energy_distance_p']:.4f} | "
                f"sub p={rec['subspace_drift_p']:.4f} | "
                f"cum p={rec['cumulative_cos_p']:.4f}"
            )

            results_sig.append(rec)
            gc.collect()

    df_sig = pd.DataFrame(results_sig)
    if df_sig.empty:
        raise ValueError(f"No significance results generated for branch {branch_name}")

    p_cols = [
        "cosine_dist_p",
        "mmd_p",
        "sliced_wasserstein_p",
        "energy_distance_p",
        "subspace_drift_p",
        "cumulative_cos_p",
    ]

    stack_index = []
    stack_vals = []
    for ridx in df_sig.index:
        for col in p_cols:
            val = df_sig.at[ridx, col]
            if pd.notna(val):
                stack_index.append((ridx, col))
                stack_vals.append(val)

    q_vals = benjamini_hochberg(stack_vals)
    for (ridx, col), q in zip(stack_index, q_vals):
        df_sig.at[ridx, col.replace("_p", "_q")] = q

    base_drift = pd.read_csv(base_drift_path)
    base_drift = normalize_drift_bins(base_drift, bin_labels)
    base_drift = base_drift[base_drift["category"].isin(categories)].copy()

    df_drift_sig = base_drift.merge(
        df_sig,
        on=["category", "bin_from", "bin_to"],
        how="left",
        validate="one_to_one",
    )

    df_drift_sig.to_csv(out_csv, index=False)

    sig_config = {
        **expected_config,
        "mmd_max_rows": MMD_MAX_ROWS,
        "ed_max_rows": ED_MAX_ROWS,
        "sw_max_rows": SW_MAX_ROWS,
        "sw_n_proj": SW_N_PROJ,
        "subspace_max_rows": SUBSPACE_MAX_ROWS,
        "subspace_n_components": SUBSPACE_N_COMPONENTS,
        "sigma_global": float(sigma_global),
        "bin_labels": list(bin_labels),
    }

    with open(sig_config_path, "w", encoding="utf-8") as f:
        json.dump(sig_config, f, indent=2)

    print(f"Saved {branch_name} significance rows -> {out_csv}")
    print(f"Saved {branch_name} config -> {sig_config_path}")

    return df_drift_sig, sig_config

# ──────────────────────────────────────────────────────────────────────
def example_key(row) -> str:
    sent = " ".join(str(row["sent"]).split())
    return f'{row["bibcode"]}|||{sent}|||{int(row["start"])}|||{int(row["end"])}'

# ──────────────────────────────────────────────────────────────────────
def exact_rank_trend_test(x, y, method="spearman", alternative="greater"):
    """
    Exact permutation test for monotonic association between x and y.
    Permutes y over fixed x.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) != len(y):
        raise ValueError("x and y must have same length")
    if len(x) < 2:
        return np.nan, np.nan, np.array([])

    if method == "spearman":
        stat_obs = spearmanr(x, y).statistic
        stat_fn = lambda a, b: spearmanr(a, b).statistic
    elif method == "kendall":
        stat_obs = kendalltau(x, y).statistic
        stat_fn = lambda a, b: kendalltau(a, b).statistic
    else:
        raise ValueError("method must be 'spearman' or 'kendall'")

    null_stats = np.array([stat_fn(x, yp) for yp in permutations(y)], dtype=float)

    if alternative == "greater":
        p = np.mean(null_stats >= stat_obs)
    elif alternative == "less":
        p = np.mean(null_stats <= stat_obs)
    elif alternative == "two-sided":
        p = np.mean(np.abs(null_stats) >= abs(stat_obs))
    else:
        raise ValueError("alternative must be 'greater', 'less', or 'two-sided'")

    return float(stat_obs), float(p), null_stats


def compact_bin_label(bin_label: str) -> str:
    start, end = bin_label.split("-")
    return f"{start}-{end[-2:]}"

def transition_label(bin_from: str, bin_to: str) -> str:
    return f"{compact_bin_label(bin_from)}\n↓\n{compact_bin_label(bin_to)}"

## Load data, validate alignment, build row meta-data
---

In [ ]:
X_reduced = np.load(REDUCED_FILE)
print("X_reduced shape:", X_reduced.shape)

with open(INDEX_PATH, "r", encoding="utf-8") as f:
    index_rows = [json.loads(line) for line in f]
corpus_df = pd.DataFrame(index_rows).copy()

with open(CLASSIFIED_PATH, "r", encoding="utf-8") as f:
    classified_rows = [json.loads(line) for line in f]
classified_df = pd.DataFrame(classified_rows).copy()
classified_df["example_key"] = classified_df.apply(example_key, axis=1)

if len(corpus_df) != len(X_reduced):
    raise ValueError(f"corpus_df/X_reduced length mismatch: {len(corpus_df)} vs {len(X_reduced)}")
if len(classified_df) != len(corpus_df):
    raise ValueError(f"classified_df/corpus_df length mismatch: {len(classified_df)} vs {len(corpus_df)}")

required_index_cols = [
    "example_key", "bin", "year", "bibcode", "sent",
    "labels", "prob_A", "prob_B", "prob_C", "dominant",
]
missing_index_cols = [c for c in required_index_cols if c not in corpus_df.columns]
if missing_index_cols:
    raise ValueError(f"Aligned index missing required columns: {missing_index_cols}")

compare_cols = ["example_key", "labels", "prob_A", "prob_B", "prob_C"]
check = corpus_df[compare_cols].merge(
    classified_df[compare_cols],
    on="example_key",
    how="outer",
    suffixes=("_index", "_classified"),
    indicator=True,
)

if not (check["_merge"] == "both").all():
    bad = check.loc[check["_merge"] != "both", ["example_key", "_merge"]].head(5)
    raise ValueError("Aligned index and classified corpus disagree on row keys. Examples\n" + str(bad))

for col in ["labels", "prob_A", "prob_B", "prob_C"]:
    left = check[f"{col}_index"]
    right = check[f"{col}_classified"]
    mismatch = left.apply(tuple) != right.apply(tuple) if col == "labels" else left != right
    if mismatch.any():
        examples = check.loc[mismatch, ["example_key", f"{col}_index", f"{col}_classified"]].head(5)
        raise ValueError(f"Aligned index is stale for column {col}. Examples\n" + str(examples))

print(f"Loaded aligned corpus_df from index: {len(corpus_df):,} rows")
print("Aligned index matches current classified corpus for labels and probabilities.")
print("Using 'dominant' from embedding_label_index.jsonl for sectional significance.")


In [ ]:
# ── Build Row Meta-Data ────────────────────────────────────────────────────────────────────
row_meta, align_mode = build_row_meta(corpus_df)
print(f"Bibcode alignment mode: {align_mode}")

bin_labels_sig = sorted(row_meta["bin"].unique(), key=lambda s: int(str(s).split("-")[0]))
raw_group_rows = build_group_rows(row_meta, bin_labels_sig)
raw_row_counts = row_meta.groupby("bin")["row_id"].count().to_dict()
raw_bib_counts = row_meta.groupby("bin")["bibcode"].nunique().to_dict()

cat_group_rows, cat_row_counts, cat_bib_counts = build_category_group_rows(
    row_meta,
    bin_labels_sig,
    categories=SIG_CATEGORIES,
    use_dominant=SIG_USE_DOMINANT,
)

for b in bin_labels_sig:
    print(f"{b}: raw rows={raw_row_counts[b]:,} | raw unique_bibcodes={raw_bib_counts[b]:,}")
print(f"Total raw rows: {sum(raw_row_counts.values()):,}")

for cat in SIG_CATEGORIES:
    print(f"\nCategory {cat} ({'dominant' if SIG_USE_DOMINANT else 'multilabel'}):")
    for b in bin_labels_sig:
        print(
            f"  {b}: rows={cat_row_counts[cat][b]:,} | unique_bibcodes={cat_bib_counts[cat][b]:,}"
        )

In [ ]:
# ── Global SIGMA ────────────────────────────────────────────────────────────────────
if "GLOBAL_SIGMA_RAW" in globals():
    sigma_global = float(GLOBAL_SIGMA_RAW)
else:
    rng_sigma = np.random.default_rng(SEED)
    sample_idx = rng_sigma.choice(len(X_reduced), size=min(SIGMA_SAMPLE_ROWS, len(X_reduced)), replace=False)
    sample = X_reduced[sample_idx]
    a = sample[:min(SIGMA_PAIR_ROWS, len(sample))]
    b = sample[:min(SIGMA_PAIR_ROWS, len(sample))]
    diffs = a[:, None] - b[None, :]
    dists = np.linalg.norm(diffs.reshape(-1, diffs.shape[-1]), axis=-1)
    sigma_global = float(np.median(dists[dists > 0]))

print(f"Using sigma_global={sigma_global:.6f}")

## Main significance testing loop
---

In [ ]:
# ── Raw + category significance cache / run ─────────────────────────────────────────────────────────
common_sig_config = {
    "run_id": RUN_ID,
    "run_mode": RUN,
    "reduction": REDUCED_LABEL,
    "seed": SEED,
    "n_perm": N_PERM,
    "target_rows_default": TARGET_ROWS_DEFAULT,
    "min_rows_for_test": MIN_ROWS_FOR_TEST,
    "n_rows": int(len(X_reduced)),
    "align_mode": align_mode,
    "index_path": str(INDEX_PATH),
    "classified_path": str(CLASSIFIED_PATH),
    "reduced_file": str(REDUCED_FILE),
}

In [ ]:

df_drift_sig_raw, sig_config_raw = run_significance_branch(
    X=X_reduced,
    branch_name="raw",
    categories=("ALL",),
    group_rows_by_cat={"ALL": raw_group_rows},
    row_counts_by_cat={"ALL": raw_row_counts},
    bib_counts_by_cat={"ALL": raw_bib_counts},
    bin_labels=bin_labels_sig,
    base_drift_path=RAW_DRIFT_PATH,
    out_csv=OUT_CSV_RAW,
    sig_config_path=SIG_CONFIG_PATH_RAW,
    common_config=common_sig_config,
)

In [ ]:

df_drift_sig_cat, sig_config_cat = run_significance_branch(
    X=X_reduced,
    branch_name=SIG_LABEL_MODE,
    categories=SIG_CATEGORIES,
    group_rows_by_cat=cat_group_rows,
    row_counts_by_cat=cat_row_counts,
    bib_counts_by_cat=cat_bib_counts,
    bin_labels=bin_labels_sig,
    base_drift_path=CAT_DRIFT_PATH,
    out_csv=OUT_CSV_CAT,
    sig_config_path=SIG_CONFIG_PATH_CAT,
    common_config={
        **common_sig_config,
        "sig_label_mode": SIG_LABEL_MODE,
        "sig_use_dominant": SIG_USE_DOMINANT,
    },
)

In [ ]:

# Preserve the old name for the raw-only downstream cells.
df_drift_sig = df_drift_sig_raw.copy()

print("\nRaw significance preview:")
display(df_drift_sig_raw)

In [ ]:

print("\nCategory significance preview:")
display(df_drift_sig_cat)


## Plotting
---

In [ ]:
dfp = df_drift_sig_raw.copy()


transition_labels = [
    transition_label(a, b)
    for a, b in zip(dfp["bin_from"], dfp["bin_to"])
]
cumulative_labels = [compact_bin_label(b) for b in dfp["bin_to"]]

metric_panels = [
    {
        "col": "cumulative_cos_obs_balanced",
        "q_col": "cumulative_cos_q",
        "x": cumulative_labels,
        "title": "Cumulative Cosine Drift",
        "ylabel": "Cumulative cosine distance",
        "color": colors["A"],
        "label": "Cumulative cosine",
        "xlabel": "Later bin",
    },
    {
        "col": "cosine_dist_obs_balanced",
        "q_col": "cosine_dist_q",
        "x": transition_labels,
        "title": "Adjacent Cosine Drift",
        "ylabel": "Observed balanced effect size",
        "color": colors["A"],
        "label": "Cosine",
        "xlabel": "Transition",
    },
    {
        "col": "mmd_obs_balanced",
        "q_col": "mmd_q",
        "x": transition_labels,
        "title": "Adjacent MMD",
        "ylabel": "Observed balanced effect size",
        "color": colors["B"],
        "label": "MMD",
        "xlabel": "Transition",
    },
    {
        "col": "sliced_wasserstein_obs_balanced",
        "q_col": "sliced_wasserstein_q",
        "x": transition_labels,
        "title": "Adjacent Sliced Wasserstein",
        "ylabel": "Observed balanced effect size",
        "color": colors["A_secondary"],
        "label": "Sliced Wasserstein",
        "xlabel": "Transition",
    },
    {
        "col": "energy_distance_obs_balanced",
        "q_col": "energy_distance_q",
        "x": transition_labels,
        "title": "Adjacent Energy Distance",
        "ylabel": "Observed balanced effect size",
        "color": colors["C"],
        "label": "Energy Distance",
        "xlabel": "Transition",
    },
    {
        "col": "subspace_drift_obs_balanced",
        "q_col": "subspace_drift_q",
        "x": transition_labels,
        "title": "Adjacent Subspace Drift",
        "ylabel": "Observed balanced effect size",
        "color": colors["C_secondary"],
        "label": "Subspace",
        "xlabel": "Transition",
    },
]

fig, axes = plt.subplots(2, 3, figsize=(20, 8.5))
axes = axes.flatten()

for ax, spec in zip(axes, metric_panels):
    x = spec["x"]
    y = dfp[spec["col"]]

    ax.plot(
        x,
        y,
        marker="o",
        markersize=6,
        linewidth=2.2,
        color=spec["color"],
        label=spec["label"],
        alpha=0.95,
    )

    ax.scatter(
        x,
        y,
        s=55,
        color=spec["color"],
        zorder=5,
    )

    if spec["q_col"] in dfp.columns:
        nonsig = dfp[dfp[spec["q_col"]] >= 0.05]
        if len(nonsig):
            if spec["col"] == "cumulative_cos_obs_balanced":
                x_nonsig = [compact_bin_label(b) for b in nonsig["bin_to"]]
            else:
                x_nonsig = [transition_label(a, b) for a, b in zip(nonsig["bin_from"], nonsig["bin_to"])]

            ax.scatter(
                x_nonsig,
                nonsig[spec["col"]],
                s=90,
                facecolor=theme["bg"],
                edgecolor=theme["fg"],
                linewidth=1.2,
                zorder=6,
            )

    ax.set_title(spec["title"], pad=10)
    ax.set_ylabel(spec["ylabel"])
    ax.set_xlabel(spec["xlabel"])
    ax.legend(
        handles=[
            Line2D([0], [0], color=spec["color"], marker="o", linewidth=2.2, label=spec["label"]),
            Line2D([0], [0], linestyle="none", marker="o", markersize=9,
                   markerfacecolor=theme["bg"], markeredgecolor=theme["fg"], label="q >= 0.05"),
        ],
        frameon=False,
        loc="best",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", rotation=0)
    ax.grid(True, alpha=0.20)

for ax in axes[len(metric_panels):]:
    ax.axis("off")

fig.suptitle(f"Significance-Aware Drift Metrics (Raw / ALL, {REDUCED_LABEL}, {RUN})", y=0.98)
plt.tight_layout()


fig.savefig(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_raw_{RUN}.png", dpi=300, bbox_inches="tight")
fig.savefig(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_raw_{RUN}.pdf", bbox_inches="tight")
record_figure_output(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_raw_{RUN}.png")
record_figure_output(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_raw_{RUN}.pdf")
print(f"Saved PNG and PDF figure -> {SIGNIFICANCE_PLOT_DIR / f'adjacent_sig_grid_raw_{RUN}.png'}")

plt.show()


---

## Exact trend tests over time (ALL only)
---

In [ ]:
# ── Prep: chronology variables ────────────────────────────────────────────────────────────────────
dfp = df_drift_sig.copy()

dfp["bin_from_start"] = dfp["bin_from"].str.split("-").str[0].astype(int)
dfp["bin_to_start"] = dfp["bin_to"].str.split("-").str[0].astype(int)
dfp["transition_mid_year"] = (dfp["bin_from_start"] + dfp["bin_to_start"]) / 2

# ── Cumulative trend test ────────────────────────────────────────────────────────────────────
x_cum = dfp["bin_to_start"].to_numpy()
y_cum = dfp["cumulative_cos_obs_balanced"].to_numpy()

cum_s_rho, cum_s_p, _ = exact_rank_trend_test(
    x_cum, y_cum, method="spearman", alternative="greater"
)
cum_k_tau, cum_k_p, _ = exact_rank_trend_test(
    x_cum, y_cum, method="kendall", alternative="greater"
)

# ── Adjacent trend test ────────────────────────────────────────────────────────────────────
adj_metrics = {
    "cosine_dist_obs_balanced": "Cosine",
    "mmd_obs_balanced": "MMD",
    "sliced_wasserstein_obs_balanced": "Sliced Wasserstein",
    "energy_distance_obs_balanced": "Energy Distance",
    "subspace_drift_obs_balanced": "Subspace",
}

adj_rows = []
x_adj = dfp["transition_mid_year"].to_numpy()

for col, label in adj_metrics.items():
    y = dfp[col].to_numpy()

    rho_inc, p_inc, _ = exact_rank_trend_test(
        x_adj, y, method="spearman", alternative="greater"
    )
    rho_dec, p_dec, _ = exact_rank_trend_test(
        x_adj, y, method="spearman", alternative="less"
    )
    tau_inc, kp_inc, _ = exact_rank_trend_test(
        x_adj, y, method="kendall", alternative="greater"
    )
    tau_dec, kp_dec, _ = exact_rank_trend_test(
        x_adj, y, method="kendall", alternative="less"
    )

    adj_rows.append({
        "metric": label,
        "spearman_rho": rho_inc,
        "spearman_p_increasing": p_inc,
        "spearman_p_decreasing": p_dec,
        "kendall_tau": tau_inc,
        "kendall_p_increasing": kp_inc,
        "kendall_p_decreasing": kp_dec,
    })

df_adj_trend = pd.DataFrame(adj_rows)

# optional: correct across adjacent trend tests
trend_p_cols = [
    "spearman_p_increasing",
    "spearman_p_decreasing",
    "kendall_p_increasing",
    "kendall_p_decreasing",
]

stack_index = []
stack_vals = []
for ridx in df_adj_trend.index:
    for col in trend_p_cols:
        val = df_adj_trend.at[ridx, col]
        if pd.notna(val):
            stack_index.append((ridx, col))
            stack_vals.append(val)

trend_q_vals = benjamini_hochberg(stack_vals)
for (ridx, col), q in zip(stack_index, trend_q_vals):
    df_adj_trend.at[ridx, col.replace("_p_", "_q_")] = q

# ── Summary tables ────────────────────────────────────────────────────────────────────
df_cum_trend = pd.DataFrame([{
    "series": "Cumulative cosine drift from 1986-1990",
    "spearman_rho": cum_s_rho,
    "spearman_p_increasing": cum_s_p,
    "kendall_tau": cum_k_tau,
    "kendall_p_increasing": cum_k_p,
}])

print("Cumulative trend test")
display(df_cum_trend)

print("Adjacent drift trend tests")
display(df_adj_trend)

CUM_TREND_PATH = SIGNIFICANCE_DATA_DIR / f"trend_tests_cumulative_{RUN_ID}_{RUN}.csv"
ADJ_TREND_PATH = SIGNIFICANCE_DATA_DIR / f"trend_tests_adjacent_{RUN_ID}_{RUN}.csv"

df_cum_trend.to_csv(CUM_TREND_PATH, index=False)
df_adj_trend.to_csv(ADJ_TREND_PATH, index=False)

print(f"Saved cumulative trend results -> {CUM_TREND_PATH}")
print(f"Saved adjacent trend results -> {ADJ_TREND_PATH}")


## Plotting trends over time
---

In [ ]:
dfp = df_drift_sig_raw.copy()

def bin_center_year(bin_label: str) -> float:
    start, end = map(int, bin_label.split("-"))
    return (start + end) / 2

dfp["bin_from_center"] = dfp["bin_from"].apply(bin_center_year)
dfp["bin_to_center"] = dfp["bin_to"].apply(bin_center_year)
dfp["transition_mid_year"] = (dfp["bin_from_center"] + dfp["bin_to_center"]) / 2

metric_panels = [
    {
        "col": "cumulative_cos_obs_balanced",
        "q_col": "cumulative_cos_q",
        "x_col": "bin_to_center",
        "title": "Cumulative Cosine Drift",
        "ylabel": "Observed balanced effect size",
        "color": colors["A"],
        "label": "Cumulative cosine",
    },
    {
        "col": "cosine_dist_obs_balanced",
        "q_col": "cosine_dist_q",
        "x_col": "bin_to_center",
        "title": "Adjacent Cosine Drift",
        "ylabel": "Observed balanced effect size",
        "color": colors["A"],
        "label": "Cosine",
    },
    {
        "col": "mmd_obs_balanced",
        "q_col": "mmd_q",
        "x_col": "bin_to_center",
        "title": "Adjacent MMD",
        "ylabel": "Observed balanced effect size",
        "color": colors["B"],
        "label": "MMD",
    },
    {
        "col": "sliced_wasserstein_obs_balanced",
        "q_col": "sliced_wasserstein_q",
        "x_col": "bin_to_center",
        "title": "Adjacent Sliced Wasserstein",
        "ylabel": "Observed balanced effect size",
        "color": colors["A_secondary"],
        "label": "Sliced Wasserstein",
    },
    {
        "col": "energy_distance_obs_balanced",
        "q_col": "energy_distance_q",
        "x_col": "bin_to_center",
        "title": "Adjacent Energy Distance",
        "ylabel": "Observed balanced effect size",
        "color": colors["C"],
        "label": "Energy Distance",
    },
    {
        "col": "subspace_drift_obs_balanced",
        "q_col": "subspace_drift_q",
        "x_col": "bin_to_center",
        "title": "Adjacent Subspace Drift",
        "ylabel": "Observed balanced effect size",
        "color": colors["C_secondary"],
        "label": "Subspace",
    },
]

fig, axes = plt.subplots(2, 3, figsize=(16, 6.5))
axes = axes.flatten()

for ax, spec in zip(axes, metric_panels):
    x = dfp[spec["x_col"]]
    y = dfp[spec["col"]]

    ax.plot(
        x,
        y,
        marker="o",
        markersize=6,
        linewidth=2.2,
        color=spec["color"],
        label=spec["label"],
        alpha=0.95,
    )
    ax.scatter(
        x,
        y,
        s=55,
        color=spec["color"],
        zorder=5,
    )

    if spec["q_col"] in dfp.columns:
        nonsig = dfp[dfp[spec["q_col"]] >= 0.05]
        if len(nonsig):
            ax.scatter(
                nonsig[spec["x_col"]],
                nonsig[spec["col"]],
                s=60,
                facecolor=theme["bg"],
                edgecolor=theme["fg"],
                linewidth=1.2,
                zorder=6,
            )

    ax.set_title(spec["title"], pad=10)
    ax.set_ylabel(spec["ylabel"])
    ax.set_xlabel("Year")
    ax.legend(frameon=False, loc="best")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.20)

for ax in axes[len(metric_panels):]:
    ax.axis("off")

fig.suptitle(f"Trend Metrics by Reducer Space and Run Mode (Raw / ALL, {REDUCED_LABEL}, {RUN})", y=1.02)
plt.tight_layout()

plot_file = SIGNIFICANCE_PLOT_DIR / f"trend_metrics_grid_raw_{RUN}.png"
fig.savefig(plot_file, dpi=300, bbox_inches="tight")
record_figure_output(plot_file)
print(f"Saved figure -> {plot_file}")

plt.show()


In [ ]:
if "df_drift_sig_cat" not in globals():
    raise ValueError("Missing df_drift_sig_cat in memory.")

dfp_cat = df_drift_sig_cat.copy()
cats_present = [cat for cat in SIG_CATEGORIES if cat in set(dfp_cat["category"])]
if not cats_present:
    raise ValueError("No category significance rows available for plotting.")

metric_panels = [
    {
        "col": "cumulative_cos_obs_balanced",
        "q_col": "cumulative_cos_q",
        "title": "Cumulative Cosine Drift",
        "kind": "cumulative",
    },
    {
        "col": "cosine_dist_obs_balanced",
        "q_col": "cosine_dist_q",
        "title": "Adjacent Cosine Drift",
        "kind": "adjacent",
    },
    {
        "col": "mmd_obs_balanced",
        "q_col": "mmd_q",
        "title": "Adjacent MMD",
        "kind": "adjacent",
    },
    {
        "col": "sliced_wasserstein_obs_balanced",
        "q_col": "sliced_wasserstein_q",
        "title": "Adjacent Sliced Wasserstein",
        "kind": "adjacent",
    },
    {
        "col": "energy_distance_obs_balanced",
        "q_col": "energy_distance_q",
        "title": "Adjacent Energy Distance",
        "kind": "adjacent",
    },
    {
        "col": "subspace_drift_obs_balanced",
        "q_col": "subspace_drift_q",
        "title": "Adjacent Subspace Drift",
        "kind": "adjacent",
    },
]

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, spec in zip(axes, metric_panels):
    for cat in cats_present:
        subset = dfp_cat[dfp_cat["category"] == cat].copy()
        if spec["kind"] == "cumulative":
            x = [compact_bin_label(b) for b in subset["bin_to"]]
            xlabel = "Later bin"
        else:
            x = [transition_label(a, b) for a, b in zip(subset["bin_from"], subset["bin_to"])]
            xlabel = "Transition"

        y = subset[spec["col"]]
        label = CAT_LABELS.get(cat, cat)
        color = colors[cat]

        ax.plot(
            x,
            y,
            marker="o",
            markersize=5,
            linewidth=2.0,
            color=color,
            label=label,
            alpha=0.95,
        )
        ax.scatter(x, y, s=45, color=color, zorder=5)

        if spec["q_col"] in subset.columns:
            nonsig = subset[subset[spec["q_col"]] >= 0.05]
            if len(nonsig):
                if spec["kind"] == "cumulative":
                    x_nonsig = [compact_bin_label(b) for b in nonsig["bin_to"]]
                else:
                    x_nonsig = [transition_label(a, b) for a, b in zip(nonsig["bin_from"], nonsig["bin_to"])]
                ax.scatter(
                    x_nonsig,
                    nonsig[spec["col"]],
                    s=80,
                    facecolor=theme["bg"],
                    edgecolor=color,
                    linewidth=1.2,
                    zorder=6,
                )

    ax.set_title(spec["title"], pad=10)
    ax.set_ylabel("Observed balanced effect size")
    ax.set_xlabel(xlabel)
    ax.grid(True, alpha=0.20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, loc="best")

fig.suptitle(
    f"Significance-Aware Drift Metrics ({SIG_LABEL_MODE.title()} labels: {', '.join(cats_present)}, {REDUCED_LABEL}, {RUN})",
    y=1.01,
)
plt.tight_layout()

fig.savefig(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_{SIG_LABEL_MODE}_{RUN}.png", dpi=300, bbox_inches="tight")
fig.savefig(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_{SIG_LABEL_MODE}_{RUN}.pdf", bbox_inches="tight")
record_figure_output(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_{SIG_LABEL_MODE}_{RUN}.png")
record_figure_output(SIGNIFICANCE_PLOT_DIR / f"adjacent_sig_grid_{SIG_LABEL_MODE}_{RUN}.pdf")
print(f"Saved PNG and PDF figure -> {SIGNIFICANCE_PLOT_DIR / f'adjacent_sig_grid_{SIG_LABEL_MODE}_{RUN}.png'}")

plt.show()
